In [1]:
import shutil
import os

src = "/kaggle/input/datasets/saisandeepn/deep-fashion-crops/crops"
dst = "/kaggle/working"
for item in os.listdir(src):
    s = os.path.join(src, item)
    d = os.path.join(dst, item)
    
    if os.path.isdir(s):
        shutil.copytree(s, d)
    else:
        shutil.copy2(s, d)

print("All contents moved to working root!")

All contents moved to working root!


In [2]:
# ════════════════════════════════════════════════════════
# CELL 1 — Mount + paths
# ════════════════════════════════════════════════════════
#from google.colab import drive
#drive.mount('/content/drive')

import os, json
PROJ = '/kaggle/working/'
!pip install transformers accelerate -q

In [3]:
# ════════════════════════════════════════════════════════
# CELL 2 — Load BLIP-2 (FROZEN — never update weights)
#
# PDF says: "BLIP-2 and YOLO stay frozen"
# We use blip2-opt-2.7b — fits on Colab T4 in float16
# ════════════════════════════════════════════════════════
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

blip2 = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16,
    device_map={"": 0}   # pins ALL submodules to cuda:0
)
blip2.eval()

# Confirm all params frozen
for p in blip2.parameters():
    p.requires_grad = False

print("BLIP-2 loaded and frozen ✓")

Device: cuda


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

BLIP-2 loaded and frozen ✓


In [4]:
# ════════════════════════════════════════════════════════
# CELL 3 — Caption generation function (PATH FIXED)
# ════════════════════════════════════════════════════════
import json
import torch
from PIL import Image

# ⚠️ PASTE YOUR EXACT KAGGLE DATASET PATH HERE ⚠️
# It should look something like: '/kaggle/input/agri-lend-deepfashion-crops'
NB1_INPUT = '/kaggle/input/datasets/saisandeepn/deep-fashion-crops' 

# Prompt steers BLIP-2 to produce product-style descriptions
PROMPT = "Question: Describe this clothing item including color, style, fit, and fabric. Answer:"

@torch.no_grad()
def generate_caption(crop_path: str) -> str:
    img    = Image.open(crop_path).convert('RGB')
    inputs = processor(
        images=img,
        text=PROMPT,
        return_tensors="pt"
    ).to(device, torch.float16)

    out = blip2.generate(**inputs, max_new_tokens=60, num_beams=3)
    caption = processor.decode(out[0], skip_special_tokens=True).strip()

    # Explicitly free GPU tensors to prevent Out of Memory errors
    del inputs, out
    torch.cuda.empty_cache()

    if "Answer:" in caption:
        caption = caption.split("Answer:")[-1].strip()

    return caption


# ── The Path Translation Fix ─────────────────────────────
# 1. Load the raw JSON with the old paths
raw_meta_gt = json.load(open(f'{NB1_INPUT}/crop_meta_gt.json'))

# 2. Translate the paths to point to our new Kaggle Input folder
crop_meta_gt = {}
for key, old_path in raw_meta_gt.items():
    # Swap out the old working directory for our new dataset location
    new_path = old_path.replace('/kaggle/working', NB1_INPUT)
    crop_meta_gt[key] = new_path

# 3. Quick test
test_path = list(crop_meta_gt.values())[0]
print(f"Reading from: {test_path}")
print(f"Test caption: {generate_caption(test_path)}")

Reading from: /kaggle/input/datasets/saisandeepn/deep-fashion-crops/crops/gt/img__WOMEN__Dresses__id_00000002__02_1_front.jpg
Test caption: black floral print dress


In [5]:
# ════════════════════════════════════════════════════════
# CELL 4 — BLIP-2 Batch Captioning (CLEAN UI + OPTIMIZED)
# ════════════════════════════════════════════════════════
import torch, json, os, warnings
from PIL import Image
from tqdm.auto import tqdm  # Auto chooses the best UI for Kaggle

# >>> CRITICAL UI FIX: Mute all annoying AI warnings that break the progress bar <<<
warnings.filterwarnings("ignore")
import transformers
transformers.logging.set_verbosity_error()

# 1. EXACT KAGGLE PATHS
NB1_INPUT = '/kaggle/input/datasets/saisandeepn/deep-fashion-crops'
PROJ = '/kaggle/working'
CAPTION_FILE = f'{PROJ}/captions/captions.json'

os.makedirs(f'{PROJ}/captions', exist_ok=True)

BATCH_SIZE = 8   

# 2. CHECKPOINT RESUME LOGIC
if os.path.exists(CAPTION_FILE):
    captions = json.load(open(CAPTION_FILE))
    print(f"Resuming — already captioned: {len(captions):,}")
else:
    captions = {}

remaining = [(k, v) for k, v in crop_meta_gt.items() if k not in captions]
print(f"Remaining to caption: {len(remaining):,}\n")

PROMPT = "Question: Describe this clothing item including color, style, fit, and fabric. Answer:"
errors = 0

# 3. THE SILENT BATCH PROCESSING LOOP
# We use a progress bar object so we can update it cleanly
pbar = tqdm(total=len(remaining), desc="Captions Generated", unit="img")

for batch_start in range(0, len(remaining), BATCH_SIZE):
    batch = remaining[batch_start: batch_start + BATCH_SIZE]
    imgs = []
    keys = []

    for i, (img_rel_path, crop_path) in enumerate(batch):
        try:
            actual_path = crop_path.replace('/kaggle/working', NB1_INPUT)
            img = Image.open(actual_path).convert('RGB')
            imgs.append(img)
            keys.append(img_rel_path)
        except Exception:
            captions[img_rel_path] = ""
            errors += 1

    if not imgs:
        pbar.update(len(batch))
        continue

    try:
        inputs = processor(
            images=imgs,
            text=[PROMPT] * len(imgs),
            return_tensors="pt",
            padding=True
        ).to(device, torch.float16)

        with torch.no_grad():
            out = blip2.generate(**inputs, max_new_tokens=60, num_beams=3)

        for i, (key, ids) in enumerate(zip(keys, out)):
            caption = processor.decode(ids, skip_special_tokens=True).strip()
            if "Answer:" in caption:
                caption = caption.split("Answer:")[-1].strip()
            captions[key] = caption

        del inputs, out
        torch.cuda.empty_cache()

    except Exception:
        for key in keys:
            captions[key] = ""
        errors += len(keys)
        torch.cuda.empty_cache()

    # Update progress bar smoothly and display error count dynamically
    pbar.update(len(batch))
    if errors > 0:
        pbar.set_postfix({"errors": errors})

    # Save checkpoint silently
    if len(captions) % 500 < BATCH_SIZE:
        with open(CAPTION_FILE, 'w') as f:
            json.dump(captions, f)

pbar.close()

# 4. FINAL SAVE
with open(CAPTION_FILE, 'w') as f:
    json.dump(captions, f)

empty = sum(1 for v in captions.values() if not v)
print(f"\n✅ Done! Total: {len(captions):,} | Empty: {empty:,} | Errors: {errors:,}")
print(f"Saved → {CAPTION_FILE}")

Remaining to caption: 38,494



Captions Generated:   0%|          | 0/38494 [00:00<?, ?img/s]


✅ Done! Total: 38,494 | Empty: 19 | Errors: 0
Saved → /kaggle/working/captions/captions.json


In [7]:
# ════════════════════════════════════════════════════════
# CELL 5 — QC: spot-check 6 random captions
# ════════════════════════════════════════════════════════
import random
from PIL import Image
import matplotlib.pyplot as plt

captions     = json.load(open(f'{PROJ}/captions/captions.json'))
crop_meta_gt = json.load(open(f'{PROJ}/crop_meta_gt.json'))

sample_keys = random.sample(list(captions.keys()), 6)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, key in zip(axes.flatten(), sample_keys):
    img = Image.open(crop_meta_gt[key]).convert('RGB')
    ax.imshow(img)
    cap = captions[key]
    ax.set_title(cap[:80] + ('...' if len(cap) > 80 else ''),
                 fontsize=7, wrap=True)
    ax.axis('off')
plt.suptitle("BLIP-2 captions QC", fontsize=11)
plt.tight_layout(); plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/crop_meta_gt.json'

In [8]:
import zipfile
import os

zip_name = '/kaggle/working/safe_captions.zip'
caption_file = '/kaggle/working/captions/captions.json'

print("Zipping the BLIP-2 captions...")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    if os.path.exists(caption_file):
        # Keeps the folder structure clean inside the zip
        zf.write(caption_file, 'captions/captions.json')
        print(f"✅ Successfully zipped: {caption_file}")
    else:
        print("❌ Error: captions.json not found! Did the loop finish?")

print(f"\nDone! Download '{zip_name}' from the right sidebar. It should be tiny!")

Zipping the BLIP-2 captions...
✅ Successfully zipped: /kaggle/working/captions/captions.json

Done! Download '/kaggle/working/safe_captions.zip' from the right sidebar. It should be tiny!
